# nuScenes Standalone Visualization (scene-0523 & scene-0916)
Front camera + LiDAR + 3D BBox visualization **without nuscenes-devkit**.  
Uses only the extracted mini-dataset in `./data/` (no full nuScenes needed).

- **scene-0523**: Oncoming bus, parked car on center divider, pedestrian, roundabout, parked trucks
- **scene-0916**: Parking lot, bicycle rack, parked bicycles, bus, many peds, parked scooters, parked motorcycle

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from collections import defaultdict

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Extracted mini-dataset (relative path)
DATAROOT = os.path.join(os.path.dirname(os.path.abspath('.')), 'nuscenes_vis', 'data') \
    if not os.path.exists('./data') else './data'
# Also try the absolute path as fallback
if not os.path.exists(os.path.join(DATAROOT, 'v1.0-mini-extract')):
    DATAROOT = '/home/dohyun/e2e/nuscenes_vis/data'
VERSION = 'v1.0-mini-extract'

print(f'DATAROOT: {DATAROOT}')
assert os.path.exists(os.path.join(DATAROOT, VERSION)), f'Data not found at {DATAROOT}/{VERSION}'

## 1. Load metadata

In [ ]:
def load_table(name):
    with open(os.path.join(DATAROOT, VERSION, f'{name}.json')) as f:
        return json.load(f)

scenes = load_table('scene')
samples = load_table('sample')
sample_datas = load_table('sample_data')
sample_anns = load_table('sample_annotation')
instances = load_table('instance')
categories = load_table('category')
calibrated_sensors = load_table('calibrated_sensor')
ego_poses = load_table('ego_pose')
sensors = load_table('sensor')

def index_by_token(records):
    return {r['token']: r for r in records}

scene_idx = index_by_token(scenes)
sample_idx = index_by_token(samples)
sd_idx = index_by_token(sample_datas)
cs_idx = index_by_token(calibrated_sensors)
ep_idx = index_by_token(ego_poses)
inst_idx = index_by_token(instances)
cat_idx = index_by_token(categories)
sensor_idx = index_by_token(sensors)

def get_category(ann):
    return cat_idx[inst_idx[ann['instance_token']]['category_token']]['name']

print(f'Loaded: {len(scenes)} scenes, {len(samples)} samples, {len(sample_anns)} annotations')
for s in scenes:
    print(f"  {s['name']}: {s['description']}")

## 2. Build lookups & pick representative frames

In [ ]:
sd_by_sample_channel = {}
for sd in sample_datas:
    if not sd['is_key_frame']:
        continue
    cs = cs_idx[sd['calibrated_sensor_token']]
    ch = sensor_idx[cs['sensor_token']]['channel']
    sd_by_sample_channel[(sd['sample_token'], ch)] = sd

anns_by_sample = defaultdict(list)
for ann in sample_anns:
    anns_by_sample[ann['sample_token']].append(ann)

def get_scene_samples(scene_token):
    sc = scene_idx[scene_token]
    result = []
    tok = sc['first_sample_token']
    while tok:
        result.append(tok)
        tok = sample_idx[tok]['next']
        if tok == '': break
    return result

# Pick middle frame of each scene as representative
chosen_tokens = []
for sc in scenes:
    sc_samples = get_scene_samples(sc['token'])
    chosen_tokens.append(sc_samples[len(sc_samples)//2])
    cats = {get_category(a) for a in anns_by_sample[chosen_tokens[-1]]}
    print(f"{sc['name']} - frame {len(sc_samples)//2}/{len(sc_samples)}")
    print(f"  Objects: {sorted(cats)}")

## 3. Coordinate transforms

In [ ]:
def quaternion_to_rotation_matrix(q):
    w, x, y, z = q
    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - w*z),     2*(x*z + w*y)],
        [2*(x*y + w*z),     1 - 2*(x*x + z*z), 2*(y*z - w*x)],
        [2*(x*z - w*y),     2*(y*z + w*x),     1 - 2*(x*x + y*y)]
    ])

def make_transform(translation, rotation):
    T = np.eye(4)
    T[:3, :3] = quaternion_to_rotation_matrix(rotation)
    T[:3, 3] = translation
    return T

def get_sensor_transform(sd_record):
    cs = cs_idx[sd_record['calibrated_sensor_token']]
    ep = ep_idx[sd_record['ego_pose_token']]
    sensor2global = make_transform(ep['translation'], ep['rotation']) @ \
                    make_transform(cs['translation'], cs['rotation'])
    return np.linalg.inv(sensor2global), cs

def get_3d_box_corners(ann):
    w, l, h = ann['size']
    x, y = l / 2, w / 2
    corners = np.array([
        [ x,  y, 0], [ x, -y, 0], [-x, -y, 0], [-x,  y, 0],
        [ x,  y, h], [ x, -y, h], [-x, -y, h], [-x,  y, h],
    ])
    corners[:, 2] -= h / 2
    R = quaternion_to_rotation_matrix(ann['rotation'])
    corners = (R @ corners.T).T + np.array(ann['translation'])
    return corners

def project_to_image(points_3d, intrinsic):
    pts = intrinsic @ points_3d.T
    pts[:2] /= pts[2:3]
    return pts[:2].T, pts[2]

print('Transforms ready.')

## 4. Data loading & drawing utilities

In [ ]:
def load_image(sd_record):
    return np.array(Image.open(os.path.join(DATAROOT, sd_record['filename'])))

def load_lidar(sd_record):
    return np.fromfile(os.path.join(DATAROOT, sd_record['filename']), dtype=np.float32).reshape(-1, 5)

CAT_COLORS = {
    'vehicle.bus.bendy': '#e6194b', 'vehicle.bus.rigid': '#e6194b',
    'vehicle.car': '#3cb44b', 'vehicle.truck': '#4363d8',
    'vehicle.trailer': '#911eb4', 'vehicle.construction': '#f58231',
    'vehicle.bicycle': '#42d4f4', 'vehicle.motorcycle': '#f032e6',
    'vehicle.emergency.police': '#bfef45', 'vehicle.emergency.ambulance': '#bfef45',
    'human.pedestrian.adult': '#fabed4', 'human.pedestrian.child': '#fabed4',
    'human.pedestrian.construction_worker': '#ffd8b1',
    'human.pedestrian.personal_mobility': '#dcbeff',
    'human.pedestrian.police_officer': '#aaffc3',
}
CAT_DISPLAY = {
    'vehicle.bus.bendy': 'Bus', 'vehicle.bus.rigid': 'Bus',
    'vehicle.car': 'Car', 'vehicle.truck': 'Truck',
    'vehicle.trailer': 'Trailer', 'vehicle.construction': 'Construction',
    'vehicle.bicycle': 'Bicycle', 'vehicle.motorcycle': 'Motorcycle',
    'vehicle.emergency.police': 'Police', 'vehicle.emergency.ambulance': 'Ambulance',
    'human.pedestrian.adult': 'Pedestrian', 'human.pedestrian.child': 'Child',
    'human.pedestrian.construction_worker': 'Worker',
    'human.pedestrian.personal_mobility': 'Scooter',
    'human.pedestrian.police_officer': 'Police Officer',
}
def get_color(cat): return CAT_COLORS.get(cat, '#808080')
def get_display_name(cat): return CAT_DISPLAY.get(cat, cat.split('.')[-1])

BOX_EDGES = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]

def clip_line_to_rect(x0, y0, x1, y1, xmin, ymin, xmax, ymax):
    INSIDE, LEFT, RIGHT, BOTTOM, TOP = 0, 1, 2, 4, 8
    def code(x, y):
        c = INSIDE
        if x < xmin: c |= LEFT
        elif x > xmax: c |= RIGHT
        if y < ymin: c |= BOTTOM
        elif y > ymax: c |= TOP
        return c
    c0, c1 = code(x0, y0), code(x1, y1)
    for _ in range(20):
        if not (c0 | c1): return x0, y0, x1, y1
        if c0 & c1: return None
        c = c0 or c1
        if c & TOP: x = x0 + (x1-x0)*(ymax-y0)/(y1-y0); y = ymax
        elif c & BOTTOM: x = x0 + (x1-x0)*(ymin-y0)/(y1-y0); y = ymin
        elif c & RIGHT: y = y0 + (y1-y0)*(xmax-x0)/(x1-x0); x = xmax
        elif c & LEFT: y = y0 + (y1-y0)*(xmin-x0)/(x1-x0); x = xmin
        if c == c0: x0, y0, c0 = x, y, code(x, y)
        else: x1, y1, c1 = x, y, code(x, y)
    return None

def draw_box_on_image(ax, corners_2d, color, linewidth=1.5, img_w=1600, img_h=900):
    margin = 50
    for i, j in BOX_EDGES:
        result = clip_line_to_rect(
            corners_2d[i,0], corners_2d[i,1], corners_2d[j,0], corners_2d[j,1],
            -margin, -margin, img_w+margin, img_h+margin)
        if result is not None:
            ax.plot([result[0], result[2]], [result[1], result[3]],
                    color=color, linewidth=linewidth)

print('Utilities ready.')

## 5. Camera + 3D BBox visualization

In [ ]:
def visualize_camera_with_boxes(sample_token, save_path=None):
    cam_sd = sd_by_sample_channel.get((sample_token, 'CAM_FRONT'))
    if cam_sd is None: print('No CAM_FRONT'); return
    img = load_image(cam_sd)
    h, w = img.shape[:2]
    global2cam, cam_cs = get_sensor_transform(cam_sd)
    intrinsic = np.array(cam_cs['camera_intrinsic'])
    
    fig, ax = plt.subplots(1, 1, figsize=(16, 9))
    ax.imshow(img); ax.set_xlim(0, w); ax.set_ylim(h, 0)
    legend_entries = {}
    for ann in anns_by_sample[sample_token]:
        cat = get_category(ann); color = get_color(cat)
        corners_cam = (global2cam @ np.hstack([get_3d_box_corners(ann), np.ones((8,1))]).T).T[:, :3]
        if np.any(corners_cam[:, 2] <= 0.5): continue
        corners_2d, depths = project_to_image(corners_cam, intrinsic)
        if corners_2d[:,0].max() < -200 or corners_2d[:,0].min() > w+200 or \
           corners_2d[:,1].max() < -200 or corners_2d[:,1].min() > h+200: continue
        draw_box_on_image(ax, corners_2d, color, img_w=w, img_h=h)
        display = get_display_name(cat)
        if display not in legend_entries: legend_entries[display] = color
        dist = np.linalg.norm(corners_cam.mean(axis=0))
        cx, cy = np.clip(corners_2d.mean(axis=0), [0, 0], [w, h])
        if 10 <= cx <= w-10 and 10 <= cy <= h-10:
            ax.text(cx, cy-8, f'{display} {dist:.0f}m', color='white', fontsize=7,
                    fontweight='bold', bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.7))
    patches = [mpatches.Patch(color=c, label=n) for n, c in legend_entries.items()]
    ax.legend(handles=patches, loc='upper right', fontsize=8)
    sc = scene_idx[sample_idx[sample_token]['scene_token']]
    ax.set_title(f"{sc['name']} - Front Camera + 3D BBox\n{sc['description']}", fontsize=12)
    ax.axis('off'); plt.tight_layout()
    if save_path: fig.savefig(save_path, bbox_inches='tight', dpi=150); print(f'Saved: {save_path}')
    plt.show()

## 6. LiDAR BEV / 3D visualization

In [ ]:
def visualize_lidar_with_boxes(sample_token, save_path=None, view='bev', xlim=(-50,50), ylim=(-50,50)):
    lidar_sd = sd_by_sample_channel.get((sample_token, 'LIDAR_TOP'))
    if lidar_sd is None: print('No LIDAR_TOP'); return
    pc = load_lidar(lidar_sd)
    global2lidar, _ = get_sensor_transform(lidar_sd)
    anns = anns_by_sample[sample_token]
    
    if view == 'bev':
        fig, ax = plt.subplots(1, 1, figsize=(12, 12))
        mask = (pc[:,0]>xlim[0])&(pc[:,0]<xlim[1])&(pc[:,1]>ylim[0])&(pc[:,1]<ylim[1])
        ax.scatter(pc[mask,0], pc[mask,1], c=pc[mask,2], cmap='viridis', s=0.3, alpha=0.5, vmin=-3, vmax=2)
        legend_entries = {}
        for ann in anns:
            cat = get_category(ann); color = get_color(cat)
            corners_lidar = (global2lidar @ np.hstack([get_3d_box_corners(ann), np.ones((8,1))]).T).T[:,:3]
            bottom = corners_lidar[:4]
            ax.add_patch(plt.Polygon(bottom[:,:2], fill=False, edgecolor=color, linewidth=1.5))
            display = get_display_name(cat)
            if display not in legend_entries: legend_entries[display] = color
            cx, cy = bottom[:,:2].mean(axis=0)
            if xlim[0]<cx<xlim[1] and ylim[0]<cy<ylim[1]:
                ax.text(cx, cy, display, color='white', fontsize=6, ha='center', va='center',
                        bbox=dict(boxstyle='round,pad=0.15', facecolor=color, alpha=0.7))
        ax.add_patch(plt.Rectangle((-1,-2), 2, 4, fill=True, facecolor='yellow', edgecolor='yellow', alpha=0.8))
        ax.text(0, 0, 'EGO', ha='center', va='center', fontsize=7, fontweight='bold')
        ax.legend(handles=[mpatches.Patch(color=c, label=n) for n,c in legend_entries.items()], loc='upper right', fontsize=8)
        ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect('equal')
        ax.set_facecolor('#1a1a2e'); fig.patch.set_facecolor('#1a1a2e')
        ax.tick_params(colors='white')
        for spine in ax.spines.values(): spine.set_color('white')
        sc = scene_idx[sample_idx[sample_token]['scene_token']]
        ax.set_title(f"{sc['name']} - LiDAR BEV\n{sc['description']}", fontsize=12, color='white')
        ax.set_xlabel('X (m)', color='white'); ax.set_ylabel('Y (m)', color='white')
    elif view == '3d':
        fig = plt.figure(figsize=(14, 10)); ax = fig.add_subplot(111, projection='3d')
        mask = (pc[:,0]>xlim[0])&(pc[:,0]<xlim[1])&(pc[:,1]>ylim[0])&(pc[:,1]<ylim[1])
        pts = pc[mask]; pts = pts[::max(1, len(pts)//30000)]
        ax.scatter(pts[:,0], pts[:,1], pts[:,2], c=pts[:,2], cmap='viridis', s=0.2, alpha=0.3, vmin=-3, vmax=2)
        legend_entries = {}
        for ann in anns:
            cat = get_category(ann); color = get_color(cat)
            cl = (global2lidar @ np.hstack([get_3d_box_corners(ann), np.ones((8,1))]).T).T[:,:3]
            for i,j in BOX_EDGES:
                ax.plot3D([cl[i,0],cl[j,0]], [cl[i,1],cl[j,1]], [cl[i,2],cl[j,2]], color=color, linewidth=1.0)
            display = get_display_name(cat)
            if display not in legend_entries: legend_entries[display] = color
        ax.legend(handles=[mpatches.Patch(color=c, label=n) for n,c in legend_entries.items()], loc='upper right', fontsize=8)
        ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_zlim(-3, 5)
        ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
        ax.view_init(elev=30, azim=-60)
        sc = scene_idx[sample_idx[sample_token]['scene_token']]
        ax.set_title(f"{sc['name']} - LiDAR 3D\n{sc['description']}", fontsize=12)
    plt.tight_layout()
    if save_path: fig.savefig(save_path, bbox_inches='tight', dpi=150, facecolor=fig.get_facecolor()); print(f'Saved: {save_path}')
    plt.show()

## 7. LiDAR projected on camera

In [ ]:
def visualize_lidar_on_image(sample_token, save_path=None):
    cam_sd = sd_by_sample_channel.get((sample_token, 'CAM_FRONT'))
    lidar_sd = sd_by_sample_channel.get((sample_token, 'LIDAR_TOP'))
    if cam_sd is None or lidar_sd is None: print('Missing data'); return
    img = load_image(cam_sd); h, w = img.shape[:2]
    pc = load_lidar(lidar_sd)
    lidar_cs = cs_idx[lidar_sd['calibrated_sensor_token']]
    lidar_ep = ep_idx[lidar_sd['ego_pose_token']]
    lidar2global = make_transform(lidar_ep['translation'], lidar_ep['rotation']) @ \
                   make_transform(lidar_cs['translation'], lidar_cs['rotation'])
    cam_cs = cs_idx[cam_sd['calibrated_sensor_token']]
    cam_ep = ep_idx[cam_sd['ego_pose_token']]
    cam2global = make_transform(cam_ep['translation'], cam_ep['rotation']) @ \
                 make_transform(cam_cs['translation'], cam_cs['rotation'])
    lidar2cam = np.linalg.inv(cam2global) @ lidar2global
    intrinsic = np.array(cam_cs['camera_intrinsic'])
    pts_cam = (lidar2cam @ np.hstack([pc[:,:3], np.ones((len(pc),1))]).T).T[:,:3]
    mask = pts_cam[:,2] > 1.0; pts_cam = pts_cam[mask]
    pts_2d, depths = project_to_image(pts_cam, intrinsic)
    mask = (pts_2d[:,0]>=0)&(pts_2d[:,0]<w)&(pts_2d[:,1]>=0)&(pts_2d[:,1]<h)
    pts_2d, depths = pts_2d[mask], depths[mask]
    fig, ax = plt.subplots(1, 1, figsize=(16, 9))
    ax.imshow(img)
    sc_plot = ax.scatter(pts_2d[:,0], pts_2d[:,1], c=depths, cmap='turbo', s=1.0, alpha=0.7, vmin=1, vmax=60)
    plt.colorbar(sc_plot, ax=ax, label='Depth (m)', shrink=0.6)
    scene = scene_idx[sample_idx[sample_token]['scene_token']]
    ax.set_title(f"{scene['name']} - LiDAR on Camera\n{scene['description']}", fontsize=12)
    ax.axis('off'); plt.tight_layout()
    if save_path: fig.savefig(save_path, bbox_inches='tight', dpi=150); print(f'Saved: {save_path}')
    plt.show()

## 8. Run all image visualizations

In [ ]:
SAVE_DIR = os.path.join(os.path.dirname(DATAROOT), 'output_standalone') \
    if os.path.exists(os.path.dirname(DATAROOT)) else './output_standalone'
os.makedirs(SAVE_DIR, exist_ok=True)

for tok in chosen_tokens:
    sc = scene_idx[sample_idx[tok]['scene_token']]
    prefix = sc['name'].replace('-', '_')
    print(f"\n{'='*60}")
    print(f"Visualizing {sc['name']}: {sc['description']}")
    print(f"{'='*60}")
    visualize_camera_with_boxes(tok, save_path=os.path.join(SAVE_DIR, f'{prefix}_cam_bbox.png'))
    visualize_lidar_with_boxes(tok, save_path=os.path.join(SAVE_DIR, f'{prefix}_lidar_bev.png'), view='bev')
    visualize_lidar_with_boxes(tok, save_path=os.path.join(SAVE_DIR, f'{prefix}_lidar_3d.png'), view='3d')
    visualize_lidar_on_image(tok, save_path=os.path.join(SAVE_DIR, f'{prefix}_lidar_on_cam.png'))

print(f'\nAll images saved to: {SAVE_DIR}')

## 9. Sequence overview (8 frames)

In [ ]:
for sc in scenes:
    all_s = get_scene_samples(sc['token'])
    n_frames = min(8, len(all_s))
    indices = np.linspace(0, len(all_s)-1, n_frames, dtype=int)
    fig, axes = plt.subplots(2, 4, figsize=(24, 10))
    for idx, ax in zip(indices, axes.flat):
        stok = all_s[idx]
        cam_sd = sd_by_sample_channel.get((stok, 'CAM_FRONT'))
        if cam_sd is None: continue
        img = load_image(cam_sd); h, w = img.shape[:2]
        global2cam, cam_cs = get_sensor_transform(cam_sd)
        intrinsic = np.array(cam_cs['camera_intrinsic'])
        ax.imshow(img); ax.set_xlim(0, w); ax.set_ylim(h, 0)
        for ann in anns_by_sample[stok]:
            cat = get_category(ann); color = get_color(cat)
            corners_cam = (global2cam @ np.hstack([get_3d_box_corners(ann), np.ones((8,1))]).T).T[:,:3]
            if np.any(corners_cam[:,2] <= 0.5): continue
            corners_2d, _ = project_to_image(corners_cam, intrinsic)
            if corners_2d[:,0].max()<-200 or corners_2d[:,0].min()>w+200 or \
               corners_2d[:,1].max()<-200 or corners_2d[:,1].min()>h+200: continue
            draw_box_on_image(ax, corners_2d, color, linewidth=1.0, img_w=w, img_h=h)
        ax.set_title(f'Frame {idx}/{len(all_s)-1}', fontsize=10); ax.axis('off')
    fig.suptitle(f"{sc['name']} - Sequence\n{sc['description']}", fontsize=14)
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, f'{sc["name"].replace("-","_")}_sequence.png'), bbox_inches='tight', dpi=150)
    plt.show()
print('Sequences saved.')

## 10. Video generation

In [ ]:
import cv2
import subprocess
import matplotlib
matplotlib.use('Agg')

def fig_to_array(fig):
    fig.canvas.draw()
    return np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy()

def render_cam_bbox_frame(sample_token, fig_w=12, fig_h=6.75):
    cam_sd = sd_by_sample_channel.get((sample_token, 'CAM_FRONT'))
    if cam_sd is None: return None
    img = load_image(cam_sd); h, w = img.shape[:2]
    global2cam, cam_cs = get_sensor_transform(cam_sd)
    intrinsic = np.array(cam_cs['camera_intrinsic'])
    fig, ax = plt.subplots(1, 1, figsize=(fig_w, fig_h), dpi=128)
    ax.imshow(img); ax.set_xlim(0, w); ax.set_ylim(h, 0)
    legend_entries = {}
    for ann in anns_by_sample[sample_token]:
        cat = get_category(ann); color = get_color(cat)
        corners_cam = (global2cam @ np.hstack([get_3d_box_corners(ann), np.ones((8,1))]).T).T[:,:3]
        if np.any(corners_cam[:,2] <= 0.5): continue
        corners_2d, _ = project_to_image(corners_cam, intrinsic)
        if corners_2d[:,0].max()<-200 or corners_2d[:,0].min()>w+200 or \
           corners_2d[:,1].max()<-200 or corners_2d[:,1].min()>h+200: continue
        draw_box_on_image(ax, corners_2d, color, img_w=w, img_h=h)
        display = get_display_name(cat)
        if display not in legend_entries: legend_entries[display] = color
        dist = np.linalg.norm(corners_cam.mean(axis=0))
        cx, cy = np.clip(corners_2d.mean(axis=0), [0,0], [w,h])
        if 10<=cx<=w-10 and 10<=cy<=h-10:
            ax.text(cx, cy-8, f'{display} {dist:.0f}m', color='white', fontsize=6,
                    fontweight='bold', bbox=dict(boxstyle='round,pad=0.15', facecolor=color, alpha=0.7))
    if legend_entries:
        ax.legend(handles=[mpatches.Patch(color=c, label=n) for n,c in legend_entries.items()], loc='upper right', fontsize=7)
    ax.axis('off'); plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    frame = fig_to_array(fig); plt.close(fig)
    return frame

def render_bev_frame(sample_token, fig_size=6.75, xlim=(-50,50), ylim=(-50,50)):
    lidar_sd = sd_by_sample_channel.get((sample_token, 'LIDAR_TOP'))
    if lidar_sd is None: return None
    pc = load_lidar(lidar_sd)
    global2lidar, _ = get_sensor_transform(lidar_sd)
    fig, ax = plt.subplots(1, 1, figsize=(fig_size, fig_size), dpi=128)
    mask = (pc[:,0]>xlim[0])&(pc[:,0]<xlim[1])&(pc[:,1]>ylim[0])&(pc[:,1]<ylim[1])
    ax.scatter(pc[mask,0], pc[mask,1], c=pc[mask,2], cmap='viridis', s=0.2, alpha=0.5, vmin=-3, vmax=2)
    for ann in anns_by_sample[sample_token]:
        cat = get_category(ann); color = get_color(cat)
        cl = (global2lidar @ np.hstack([get_3d_box_corners(ann), np.ones((8,1))]).T).T[:,:3]
        ax.add_patch(plt.Polygon(cl[:4,:2], fill=False, edgecolor=color, linewidth=1.5))
    ax.add_patch(plt.Rectangle((-1,-2), 2, 4, fill=True, facecolor='yellow', edgecolor='yellow', alpha=0.8))
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect('equal')
    ax.set_facecolor('#1a1a2e'); fig.patch.set_facecolor('#1a1a2e')
    ax.tick_params(colors='white', labelsize=7)
    for spine in ax.spines.values(): spine.set_color('white')
    ax.set_xlabel('X (m)', color='white', fontsize=8); ax.set_ylabel('Y (m)', color='white', fontsize=8)
    plt.tight_layout(pad=0.5)
    frame = fig_to_array(fig); plt.close(fig)
    return frame

def render_lidar_on_cam_frame(sample_token, fig_w=12, fig_h=6.75):
    cam_sd = sd_by_sample_channel.get((sample_token, 'CAM_FRONT'))
    lidar_sd = sd_by_sample_channel.get((sample_token, 'LIDAR_TOP'))
    if cam_sd is None or lidar_sd is None: return None
    img = load_image(cam_sd); h, w = img.shape[:2]; pc = load_lidar(lidar_sd)
    lidar_cs = cs_idx[lidar_sd['calibrated_sensor_token']]; lidar_ep = ep_idx[lidar_sd['ego_pose_token']]
    lidar2global = make_transform(lidar_ep['translation'], lidar_ep['rotation']) @ make_transform(lidar_cs['translation'], lidar_cs['rotation'])
    cam_cs = cs_idx[cam_sd['calibrated_sensor_token']]; cam_ep = ep_idx[cam_sd['ego_pose_token']]
    cam2global = make_transform(cam_ep['translation'], cam_ep['rotation']) @ make_transform(cam_cs['translation'], cam_cs['rotation'])
    lidar2cam = np.linalg.inv(cam2global) @ lidar2global
    intrinsic = np.array(cam_cs['camera_intrinsic'])
    pts_cam = (lidar2cam @ np.hstack([pc[:,:3], np.ones((len(pc),1))]).T).T[:,:3]
    mask = pts_cam[:,2]>1.0; pts_cam = pts_cam[mask]
    pts_2d, depths = project_to_image(pts_cam, intrinsic)
    mask = (pts_2d[:,0]>=0)&(pts_2d[:,0]<w)&(pts_2d[:,1]>=0)&(pts_2d[:,1]<h)
    pts_2d, depths = pts_2d[mask], depths[mask]
    fig, ax = plt.subplots(1, 1, figsize=(fig_w, fig_h), dpi=128)
    ax.imshow(img)
    ax.scatter(pts_2d[:,0], pts_2d[:,1], c=depths, cmap='turbo', s=0.8, alpha=0.7, vmin=1, vmax=60)
    ax.axis('off'); plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    frame = fig_to_array(fig); plt.close(fig)
    return frame

print('Video renderers ready.')

In [ ]:
VIDEO_DIR = os.path.join(SAVE_DIR, 'videos')
os.makedirs(VIDEO_DIR, exist_ok=True)

def write_video(scene_token, render_fn, output_path, fps=2):
    sample_tokens = get_scene_samples(scene_token)
    sc = scene_idx[scene_token]
    print(f"  {os.path.basename(output_path)}: {len(sample_tokens)} frames...", end=' ')
    writer = None
    for stok in sample_tokens:
        frame = render_fn(stok)
        if frame is None: continue
        if writer is None:
            fh, fw = frame.shape[:2]
            writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (fw, fh))
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    if writer: writer.release(); print('done')

def write_sidebyside_video(scene_token, output_path, fps=2):
    sample_tokens = get_scene_samples(scene_token)
    sc = scene_idx[scene_token]
    print(f"  {os.path.basename(output_path)}: {len(sample_tokens)} frames...", end=' ')
    writer = None
    for stok in sample_tokens:
        cam = render_cam_bbox_frame(stok); bev = render_bev_frame(stok)
        if cam is None or bev is None: continue
        ch, cw = cam.shape[:2]; bh, bw = bev.shape[:2]
        bev_r = cv2.resize(bev, (int(bw * ch / bh), ch))
        composite = np.hstack([cam, bev_r])
        if writer is None:
            h, w = composite.shape[:2]
            writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        writer.write(cv2.cvtColor(composite, cv2.COLOR_RGB2BGR))
    if writer: writer.release(); print('done')

for sc in scenes:
    prefix = sc['name'].replace('-', '_')
    print(f"\n{sc['name']}: {sc['description']}")
    write_video(sc['token'], render_cam_bbox_frame, os.path.join(VIDEO_DIR, f'{prefix}_cam_bbox.mp4'))
    write_video(sc['token'], render_lidar_on_cam_frame, os.path.join(VIDEO_DIR, f'{prefix}_lidar_on_cam.mp4'))
    write_sidebyside_video(sc['token'], os.path.join(VIDEO_DIR, f'{prefix}_sidebyside.mp4'))

# Convert to H.264
print(f"\nConverting to H.264...")
for f in sorted(os.listdir(VIDEO_DIR)):
    if f.endswith('.mp4'):
        src = os.path.join(VIDEO_DIR, f)
        dst = src.replace('.mp4', '_h264.mp4')
        subprocess.run(['ffmpeg','-y','-i',src,'-c:v','libx264','-preset','medium',
                        '-crf','23','-pix_fmt','yuv420p','-an',dst], capture_output=True)
        os.replace(dst, src)
        print(f"  {f}")

print(f"\nAll videos saved to: {VIDEO_DIR}")

## 11. Summary

In [ ]:
print(f'=== All outputs in: {SAVE_DIR} ===')
print('\n--- Images ---')
for f in sorted(os.listdir(SAVE_DIR)):
    fp = os.path.join(SAVE_DIR, f)
    if os.path.isfile(fp):
        print(f'  {f}  ({os.path.getsize(fp)/1e6:.1f} MB)')
print('\n--- Videos ---')
for f in sorted(os.listdir(VIDEO_DIR)):
    print(f'  {f}  ({os.path.getsize(os.path.join(VIDEO_DIR, f))/1e6:.1f} MB)')